# Notebook 03 — Depot-Level Model Training & Evaluation
## Machine Learning for Product Gain/Loss Prediction
**Author:** Uzoma Nnaemeka Eze | MSc Data Science | UE Potsdam

---
**Models trained:** Linear Regression, Decision Tree Regressor, Logistic Regression, Decision Tree Classifier  
**Dataset:** M2 daily depot summary — 365 records per product (AGO and PMS)  
**Split:** 80% chronological training / 20% test  
**Target (regression):** Daily Loss/Gain volume in litres  
**Target (classification):** Binary Loss Flag (1 = loss day, 0 = gain/neutral day)


In [13]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              roc_auc_score, classification_report, confusion_matrix,
                              roc_curve)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    ago = pd.read_csv('data/ago_feat.csv', parse_dates=['Date'])
    pms = pd.read_csv('data/pms_feat.csv', parse_dates=['Date'])
    print(f"AGO: {len(ago)} records | PMS: {len(pms)} records")
except:
    print("Run Notebook 02 first to generate feature files.")
    ago = pms = pd.DataFrame()

M2_FEATURES = ['Opening_Stock', 'Total_Receipts', 'Total_Deliveries',
                'Receipt_Delivery_Ratio', 'Lag1_Loss', 'Lag2_Loss',
                'Rolling3_Loss', 'DayOfWeek', 'Month']


Run Notebook 02 first to generate feature files.


## 1. Train and Evaluate All Four Models

In [14]:
def train_evaluate_m2(df, product, color):
    n     = len(df)
    split = int(n * 0.80)

    print(f"\n{'='*60}")
    print(f"  {product} — 80/20 Chronological Split")
    print(f"{'='*60}")
    print(f"Training: {split} records ({df['Date'].iloc[0].date()} to {df['Date'].iloc[split-1].date()})")
    print(f"Test    : {n-split} records ({df['Date'].iloc[split].date()} to {df['Date'].iloc[-1].date()})")
    print(f"Loss days: {df['Loss_Flag'].sum()} ({df['Loss_Flag'].mean()*100:.1f}%)")

    X      = df[M2_FEATURES]
    y_reg  = df['Loss_Gain']
    y_cls  = df['Loss_Flag']
    X_tr, X_te   = X.iloc[:split], X.iloc[split:]
    yr_tr, yr_te = y_reg.iloc[:split], y_reg.iloc[split:]
    yc_tr, yc_te = y_cls.iloc[:split], y_cls.iloc[split:]

    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)

    # Model 1 — Linear Regression
    lr = LinearRegression().fit(X_tr, yr_tr)
    lr_p = lr.predict(X_te)
    print(f"\nLinear Regression:")
    print(f"  MAE={mean_absolute_error(yr_te,lr_p):>12,.0f} L | RMSE={np.sqrt(mean_squared_error(yr_te,lr_p)):>12,.0f} L | R2={r2_score(yr_te,lr_p):.4f}")

    # Model 2 — Decision Tree Regressor
    dtr = DecisionTreeRegressor(max_depth=6, random_state=42).fit(X_tr, yr_tr)
    dtr_p = dtr.predict(X_te)
    fi = pd.Series(dtr.feature_importances_, index=M2_FEATURES).sort_values(ascending=False)
    print(f"\nDecision Tree Regressor:")
    print(f"  MAE={mean_absolute_error(yr_te,dtr_p):>12,.0f} L | RMSE={np.sqrt(mean_squared_error(yr_te,dtr_p)):>12,.0f} L | R2={r2_score(yr_te,dtr_p):.4f}")
    print(f"  Top 3: {', '.join(fi.head(3).index.tolist())}")

    # Model 3 — Logistic Regression
    log = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000).fit(X_tr_sc, yc_tr)
    log_p = log.predict(X_te_sc)
    log_prob = log.predict_proba(X_te_sc)[:,1]
    log_auc = roc_auc_score(yc_te, log_prob)
    cm_log = confusion_matrix(yc_te, log_p)
    print(f"\nLogistic Regression: AUC={log_auc:.4f}")
    print(classification_report(yc_te, log_p, target_names=['Gain/Neutral','Loss'], digits=3))
    print(f"  CM: TN={cm_log[0,0]} FP={cm_log[0,1]} FN={cm_log[1,0]} TP={cm_log[1,1]}")

    # Model 4 — Decision Tree Classifier
    dtc = DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42).fit(X_tr, yc_tr)
    dtc_p = dtc.predict(X_te)
    dtc_prob = dtc.predict_proba(X_te)[:,1]
    dtc_auc = roc_auc_score(yc_te, dtc_prob)
    cm_dtc = confusion_matrix(yc_te, dtc_p)
    print(f"\nDecision Tree Classifier: AUC={dtc_auc:.4f}")
    print(classification_report(yc_te, dtc_p, target_names=['Gain/Neutral','Loss'], digits=3))
    print(f"  CM: TN={cm_dtc[0,0]} FP={cm_dtc[0,1]} FN={cm_dtc[1,0]} TP={cm_dtc[1,1]}")

    return dict(yr_te=yr_te, yc_te=yc_te,
                lr_pred=lr_p, dtr_pred=dtr_p, feat_imp=fi,
                log_pred=log_p, log_prob=log_prob, log_auc=log_auc, cm_log=cm_log,
                dtc_pred=dtc_p, dtc_prob=dtc_prob, dtc_auc=dtc_auc, cm_dtc=cm_dtc)

results = {}
if len(ago) > 0:
    results['AGO'] = train_evaluate_m2(ago, 'AGO', '#1A5F9E')
    results['PMS'] = train_evaluate_m2(pms, 'PMS', '#C94E2D')


## 2. ROC Curves

In [15]:
if results:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('ROC Curves — Depot-Level Classification Models', fontsize=12, fontweight='bold')

    for ax, (product, color) in zip(axes, [('AGO','#1A5F9E'),('PMS','#C94E2D')]):
        r = results[product]
        for mname, prob, ls in [('Logistic Regression', r['log_prob'], '-'),
                                  ('Decision Tree Classifier', r['dtc_prob'], '--')]:
            fpr, tpr, _ = roc_curve(r['yc_te'], prob)
            auc = roc_auc_score(r['yc_te'], prob)
            ax.plot(fpr, tpr, lw=2, linestyle=ls, color=color,
                    label=f'{mname} (AUC={auc:.3f})')
        ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Random')
        ax.set_title(f'{product} — ROC Curve', fontweight='bold', color=color)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_facecolor('#FAFAFA')

    plt.tight_layout()
    plt.savefig('reports/fig_roc_depot_level.png', dpi=150, bbox_inches='tight')
    plt.show()


## 3. Pareto Analysis

In [16]:
if len(ago) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Pareto Analysis — Loss Event Concentration (AGO and PMS)', fontsize=12, fontweight='bold')

    for ax, (product, df_raw, color) in zip(axes, [('AGO',ago,'#1A5F9E'),('PMS',pms,'#C94E2D')]):
        losses = df_raw[df_raw['Loss_Gain']<0].copy()
        losses['Abs'] = losses['Loss_Gain'].abs()
        total = losses['Abs'].sum()
        ls = losses.sort_values('Abs', ascending=False).reset_index(drop=True)
        ls['CumPct'] = ls['Abs'].cumsum() / total * 100
        n80 = (ls['CumPct'] <= 80).sum()

        ax2 = ax.twinx()
        ax.bar(range(len(ls)), ls['Abs']/1000, color=color, alpha=0.7, width=0.9)
        ax2.plot(range(len(ls)), ls['CumPct'], color='#E67E22', lw=2, marker='o', markersize=3)
        ax2.axhline(80, color='red', lw=1.5, linestyle='--')
        ax.axvline(n80-0.5, color='red', lw=1.5, linestyle='--')

        ax.set_title(f'{product} — Top {n80} events ({n80/len(ls)*100:.0f}%) = 80% of loss',
                     fontweight='bold', color=color)
        ax.set_xlabel('Loss Events (ranked by magnitude)')
        ax.set_ylabel('Loss Volume (000 litres)', color=color)
        ax2.set_ylabel('Cumulative % of Total Loss', color='#E67E22')
        ax2.set_ylim(0, 110)
        ax.grid(axis='y', alpha=0.3)
        ax.set_facecolor('#FAFAFA')

    plt.tight_layout()
    plt.savefig('reports/fig_pareto_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
